## 라이브러리 및 상수 선언

In [2]:
# pip install google_play_scraper

In [3]:
from google_play_scraper import app, reviews, Sort
import time
import pandas as pd

In [4]:
PACKAGE_LIST = ['com.sampleapp','com.coupang.mobile.eats','com.fineapp.yogiyo','com.shinhan.o2o'] # 배민, 쿠팡이츠, 요기요, 땡겨요
PACKAGE_NAME = ['배달의민족', '쿠팡이츠', '요기요', '땡겨요']
PACKAGE_NUM = 3
NUM_DATA = 10000

## 크롤링

In [5]:
all_reviews = []
token = None
seen = set()

In [6]:
while True: # 한번의 호출가능한 수가 한정되어있으므로 반복해야함. count=10,000이라고해서 10,000개 가져오는거 아님.
    batch, token = reviews(
        PACKAGE_LIST[PACKAGE_NUM],
        lang="ko", # 언어
        country="kr", # 국가
        count=200,
        sort=Sort.NEWEST,
        continuation_token=token
    )
    # 중복 제거 (수집 중 리뷰가 추가되어 중복된 리뷰가 들어갈 수 있음.)
    for r in batch:
        rid = r.get("reviewId")
        if rid not in seen:
            seen.add(rid)
            all_reviews.append(r)

    all_reviews.extend(batch)
    # print(type(review[0]))
    # print(type(review[0][0]))
    # print(len(review[0]))
    # print(review[0][0].keys())
    # ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

    if token is None or len(all_reviews) >= NUM_DATA:
        all_reviews = all_reviews[:NUM_DATA]
        break

    time.sleep(0.5) # 너무 빠르게 호출하면 에러 발생할 수 있음.

In [7]:
print("수집한 리뷰 수:", len(all_reviews))

수집한 리뷰 수: 10000


## 간단한 EDA

In [8]:
df = pd.DataFrame(all_reviews)

In [9]:
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,1b89637c-2eb4-4173-a797-6d10c7387a8f,캡틴코리아,https://play-lh.googleusercontent.com/a/ACg8oc...,품목도 많이 없고 배달비도 싼것도 아님,3,0,1.9.3,2025-12-14 12:10:02,None,NaT,1.9.3
1,7ffdac55-77db-4286-89e5-f279b971b8fc,이은정,https://play-lh.googleusercontent.com/a/ACg8oc...,도우가 깔끔하고 맛있어요,5,0,1.9.3,2025-12-14 12:03:00,None,NaT,1.9.3
2,0f494390-ae86-4c6e-87ef-88624b04bd5d,고현,https://play-lh.googleusercontent.com/a/ACg8oc...,가격대비 만족합니다,4,0,1.9.3,2025-12-14 09:34:27,None,NaT,1.9.3
3,d65561bf-bf99-44ea-9807-0414d2194d1e,황남석,https://play-lh.googleusercontent.com/a-/ALV-U...,신규입점 많이 해주세요,5,0,1.9.3,2025-12-13 23:46:40,None,NaT,1.9.3
4,8cf95178-459a-4c11-99ac-ae8f4d642628,정대웅,https://play-lh.googleusercontent.com/a-/ALV-U...,지역화폐(카드)로 결제가 되니 편리하고 좋습니다.,5,0,1.9.3,2025-12-13 22:52:00,None,NaT,1.9.3


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              10000 non-null  object        
 1   userName              10000 non-null  object        
 2   userImage             10000 non-null  object        
 3   content               10000 non-null  object        
 4   score                 10000 non-null  int64         
 5   thumbsUpCount         10000 non-null  int64         
 6   reviewCreatedVersion  9408 non-null   object        
 7   at                    10000 non-null  datetime64[ns]
 8   replyContent          330 non-null    object        
 9   repliedAt             330 non-null    datetime64[ns]
 10  appVersion            9408 non-null   object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 859.5+ KB


In [11]:
df['reviewCreatedVersion'].unique()

array(['1.9.3', None, '1.5.3', '1.7.6', '1.9.1', '1.9.2', '1.8.7',
       '1.8.1', '1.7.3', '1.9.0', '1.8.9', '1.8.8', '1.8.6', '1.8.3',
       '1.8.5', '1.8.4', '1.7.7', '1.7.5', '1.4.7', '1.8.0', '1.7.8',
       '1.5.9', '1.6.9', '1.4.1', '1.7.4', '1.6.2', '1.7.2', '1.6.8',
       '1.6.6', '1.7.9', '1.8.2', '1.5.8', '1.7.0', '1.4.9', '1.7.1',
       '1.6.5', '1.6.3', '1.2.6', '1.6.7'], dtype=object)

In [12]:
df['appVersion'].unique()

array(['1.9.3', None, '1.5.3', '1.7.6', '1.9.1', '1.9.2', '1.8.7',
       '1.8.1', '1.7.3', '1.9.0', '1.8.9', '1.8.8', '1.8.6', '1.8.3',
       '1.8.5', '1.8.4', '1.7.7', '1.7.5', '1.4.7', '1.8.0', '1.7.8',
       '1.5.9', '1.6.9', '1.4.1', '1.7.4', '1.6.2', '1.7.2', '1.6.8',
       '1.6.6', '1.7.9', '1.8.2', '1.5.8', '1.7.0', '1.4.9', '1.7.1',
       '1.6.5', '1.6.3', '1.2.6', '1.6.7'], dtype=object)

In [13]:
(df['reviewCreatedVersion'].fillna('MISSING')  == df['appVersion'].fillna('MISSING')).sum() # nan == nan 은 false이므로 missing으로 결측치 처리 후 비교

np.int64(10000)

In [14]:
df['app'] = PACKAGE_NAME[PACKAGE_NUM]
df['platform'] = 'playstore'
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app,platform
0,1b89637c-2eb4-4173-a797-6d10c7387a8f,캡틴코리아,https://play-lh.googleusercontent.com/a/ACg8oc...,품목도 많이 없고 배달비도 싼것도 아님,3,0,1.9.3,2025-12-14 12:10:02,None,NaT,1.9.3,땡겨요,playstore
1,7ffdac55-77db-4286-89e5-f279b971b8fc,이은정,https://play-lh.googleusercontent.com/a/ACg8oc...,도우가 깔끔하고 맛있어요,5,0,1.9.3,2025-12-14 12:03:00,None,NaT,1.9.3,땡겨요,playstore
2,0f494390-ae86-4c6e-87ef-88624b04bd5d,고현,https://play-lh.googleusercontent.com/a/ACg8oc...,가격대비 만족합니다,4,0,1.9.3,2025-12-14 09:34:27,None,NaT,1.9.3,땡겨요,playstore
3,d65561bf-bf99-44ea-9807-0414d2194d1e,황남석,https://play-lh.googleusercontent.com/a-/ALV-U...,신규입점 많이 해주세요,5,0,1.9.3,2025-12-13 23:46:40,None,NaT,1.9.3,땡겨요,playstore
4,8cf95178-459a-4c11-99ac-ae8f4d642628,정대웅,https://play-lh.googleusercontent.com/a-/ALV-U...,지역화폐(카드)로 결제가 되니 편리하고 좋습니다.,5,0,1.9.3,2025-12-13 22:52:00,None,NaT,1.9.3,땡겨요,playstore


app : 배달앱 이름

platform : 스토어 종류(플레이스토어/앱스토어)

reviewId : id(중복비교를 위해 사용)

userName	: 리뷰작성한 사용자 이름

userImage : 리뷰작성한 사용자 프로필 이미지

content : 리뷰

score : 별점

thumbsUpCount : 좋아요수

reviewCreatedVersion : 리뷰가 작성될 당시 사용자가 쓰고 있던 앱 버전

at : 리뷰 작성 날짜

replyContent : 리뷰에 대해 앱 운영사(개발사)가 남긴 공식 답변 내용

repliedAt : 답글을 남긴 날짜

appVersion : 리뷰 수집 당시 구글 플레이에 노출되는 현재 앱 버전 정보

reviewCreatedVersion랑 appVersion은 다를 수 있지만 수집된 데이터에 한해서는 완전히 같은 정보를 가지고 있음.

저장할 컬럼 [app, platform, reviewId, userName, content, score, thumbsUpCount, at]

## CSV 파일 저장

In [15]:
columns = ['app', 'platform', 'reviewId', 'userName', 'content', 'score', 'thumbsUpCount', 'at']

In [16]:
df[columns].to_csv('reviews.csv', index=False, encoding="utf-8-sig") # utf-8-sig:윈도우+엑셀에서 한글 깨짐 방지